In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

data_dir = Path("../data")
files = sorted(data_dir.rglob("raw_*.csv"))

print(f"Found {len(files)} asset files\n")

series_list = []

for path in files:
    asset = path.stem.replace("raw_", "")

    df = pd.read_csv(path, usecols=["Date", "Close"])
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date")

    s = df.set_index("Date")["Close"]

    n_negative = (s < 0).sum()
    if n_negative > 0:
        print(f"  [{asset}] {n_negative} negative price(s) detected — forward-filled")

    s = s.where(s > 0, np.nan).ffill()
    log_returns = np.log(s / s.shift(1)).rename(asset)

    series_list.append(log_returns)

print(f"\nLoaded {len(series_list)} assets")

merged = pd.concat(series_list, axis=1, sort=False)
print(f"Merged shape before dropna: {merged.shape}")
print(f"Total NaNs before dropna: {merged.isna().sum().sum()}")

merged = merged.dropna()
print(f"Merged shape after dropna:  {merged.shape}")

merged = merged.reset_index()
merged = merged.sort_values("Date").reset_index(drop=True)

print(f"\nDate range: {merged['Date'].min()} → {merged['Date'].max()}")
print(f"Assets: {[c for c in merged.columns if c != 'Date']}")

merged["Date"] = merged["Date"].dt.strftime("%Y-%m-%d")
merged.to_csv(data_dir / "all_assets_log_returns.csv", index=False)

print(f"\nSaved → {data_dir / 'all_assets_log_returns.csv'}")
print(f"Final shape: {merged.shape[0]} trading days × {merged.shape[1] - 1} assets (+1 Date column)")

merged.head()


# Windowing, Macro Alignment, Normalization & Train/Val/Test Split

For the conditional GAN:
- Sliding windows of 63 daily returns (shape: `n_windows × 63 × 30`)
- Each window matched to the most recent weekly macro snapshot (shape: `n_windows × 8`)
- Temporal 80/10/10 train/val/test split
- Z-score normalization fitted on train only
- Scalers saved for denormalization of generated samples

In [ ]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd

data_dir = Path("../data")
processed_dir = data_dir / "processed"
processed_dir.mkdir(exist_ok=True)

WINDOW = 63   # trading days per window (~3 months)
STRIDE = 1    # slide by 1 day
TRAIN_FRAC = 0.80
VAL_FRAC   = 0.10
# test gets the remaining 10%

# ── 1. Load data ──────────────────────────────────────────────────────────────
returns = pd.read_csv(data_dir / "all_assets_log_returns.csv", parse_dates=["Date"])
returns = returns.sort_values("Date").reset_index(drop=True)

macro = pd.read_csv(data_dir / "macro" / "macro_conditioning_features.csv", parse_dates=["Date"])
macro = macro.sort_values("Date").set_index("Date")

# Align macro to the same start as returns
overlap_start = returns["Date"].min()
macro = macro[macro.index >= overlap_start]

asset_cols = [c for c in returns.columns if c != "Date"]
macro_cols  = macro.columns.tolist()

print(f"Returns : {returns.shape}  ({returns['Date'].min().date()} → {returns['Date'].max().date()})")
print(f"Macro   : {macro.shape}  ({macro.index.min().date()} → {macro.index.max().date()})")
print(f"Assets  : {len(asset_cols)}")
print(f"Macro features: {macro_cols}")


In [ ]:
# ── 2. Sliding windows + macro alignment ─────────────────────────────────────
ret_values = returns[asset_cols].values   # (T, 30)
dates      = returns["Date"].values        # (T,)

windows      = []
macro_snaps  = []
window_dates = []   # start date of each window (for split)

for i in range(0, len(ret_values) - WINDOW + 1, STRIDE):
    window = ret_values[i : i + WINDOW]       # (63, 30)
    start_date = dates[i]

    # most recent weekly macro on or before start_date
    past_macro = macro[macro.index <= start_date]
    if past_macro.empty:
        continue                              # no macro coverage yet — skip
    macro_vec = past_macro.iloc[-1].values   # (8,)

    windows.append(window)
    macro_snaps.append(macro_vec)
    window_dates.append(start_date)

X     = np.array(windows,     dtype=np.float32)   # (N, 63, 30)
M     = np.array(macro_snaps, dtype=np.float32)   # (N, 8)
dates = np.array(window_dates)                     # (N,)

print(f"Total windows : {len(X)}")
print(f"X shape       : {X.shape}   (windows × days × assets)")
print(f"M shape       : {M.shape}   (windows × macro features)")
print(f"Date range    : {dates[0]} → {dates[-1]}")


In [ ]:
# ── 3. Temporal train / val / test split ─────────────────────────────────────
N = len(X)
n_train = int(N * TRAIN_FRAC)
n_val   = int(N * VAL_FRAC)

X_train, M_train = X[:n_train],           M[:n_train]
X_val,   M_val   = X[n_train:n_train+n_val], M[n_train:n_train+n_val]
X_test,  M_test  = X[n_train+n_val:],    M[n_train+n_val:]

print(f"Train : {X_train.shape}  macro {M_train.shape}  dates {dates[0]} → {dates[n_train-1]}")
print(f"Val   : {X_val.shape}    macro {M_val.shape}    dates {dates[n_train]} → {dates[n_train+n_val-1]}")
print(f"Test  : {X_test.shape}   macro {M_test.shape}   dates {dates[n_train+n_val]} → {dates[-1]}")


In [ ]:
# ── 4. Z-score normalization (fit on train only) ──────────────────────────────

# Returns: fit per asset across (n_windows × n_days) flattened
ret_mean = X_train.reshape(-1, X_train.shape[-1]).mean(axis=0)  # (30,)
ret_std  = X_train.reshape(-1, X_train.shape[-1]).std(axis=0)   # (30,)
ret_std  = np.where(ret_std == 0, 1.0, ret_std)                 # avoid div by 0

X_train_n = (X_train - ret_mean) / ret_std
X_val_n   = (X_val   - ret_mean) / ret_std
X_test_n  = (X_test  - ret_mean) / ret_std

# Macro: fit per feature
mac_mean = M_train.mean(axis=0)   # (8,)
mac_std  = M_train.std(axis=0)    # (8,)
mac_std  = np.where(mac_std == 0, 1.0, mac_std)

M_train_n = (M_train - mac_mean) / mac_std
M_val_n   = (M_val   - mac_mean) / mac_std
M_test_n  = (M_test  - mac_mean) / mac_std

print("Returns normalization (per asset):")
print(f"  mean range : [{ret_mean.min():.6f}, {ret_mean.max():.6f}]")
print(f"  std  range : [{ret_std.min():.6f},  {ret_std.max():.6f}]")
print(f"  X_train_n  : mean={X_train_n.mean():.4f}  std={X_train_n.std():.4f}")

print("\nMacro normalization (per feature):")
print(f"  mean range : [{mac_mean.min():.6f}, {mac_mean.max():.6f}]")
print(f"  std  range : [{mac_std.min():.6f},  {mac_std.max():.6f}]")
print(f"  M_train_n  : mean={M_train_n.mean():.4f}  std={M_train_n.std():.4f}")


In [ ]:
# ── 5. Save everything ────────────────────────────────────────────────────────

# Return windows
np.save(processed_dir / "X_train.npy", X_train_n)
np.save(processed_dir / "X_val.npy",   X_val_n)
np.save(processed_dir / "X_test.npy",  X_test_n)

# Macro conditioning vectors
np.save(processed_dir / "macro_train.npy", M_train_n)
np.save(processed_dir / "macro_val.npy",   M_val_n)
np.save(processed_dir / "macro_test.npy",  M_test_n)

# Scalers (needed to denormalize generated samples)
scalers = {
    "ret_mean":  ret_mean,
    "ret_std":   ret_std,
    "mac_mean":  mac_mean,
    "mac_std":   mac_std,
    "asset_cols": asset_cols,
    "macro_cols": macro_cols,
    "window":    WINDOW,
}
with open(processed_dir / "scalers.pkl", "wb") as f:
    pickle.dump(scalers, f)

print("Saved to", processed_dir)
print(f"  X_train.npy      : {X_train_n.shape}")
print(f"  X_val.npy        : {X_val_n.shape}")
print(f"  X_test.npy       : {X_test_n.shape}")
print(f"  macro_train.npy  : {M_train_n.shape}")
print(f"  macro_val.npy    : {M_val_n.shape}")
print(f"  macro_test.npy   : {M_test_n.shape}")
print(f"  scalers.pkl      : ret_mean/std (30,) · mac_mean/std (8,)")
